In [1]:
!pip install -q rdkit catboost lightgbm xgboost

import os
import sys
import time
import signal
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Scikit-Learn
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import RobustScaler, QuantileTransformer, MinMaxScaler
from sklearn.linear_model import Ridge, BayesianRidge, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

# Boosting Frameworks
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# Cheminformatics (RDKit)
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import (AllChem, MACCSkeys, Descriptors, Crippen,
                        Lipinski, rdMolDescriptors, Descriptors3D)
from rdkit.Chem.EState import EState_VSA, EState as ES
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# 1. GLOBAL SETTINGS
# -----------------------------------------------------------------------------
warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

SEED = 2024  # Optimal seed for structural initialization
N_SPLITS = 5
N_BITS = 2048
TARGETS = ['egc', 'egb', 'ei', 'eea', 'eps', 'nc', 'tg']
NUM_TARGETS = len(TARGETS)

COMPUTE_3D = True
MAX_HEAVY_FOR_3D = 80

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def rmse(yt, yp):
    return float(np.sqrt(mean_squared_error(yt, yp)))

# -----------------------------------------------------------------------------
# 2. SEQUENCE PARSING & STAR SUBSTITUTION
# -----------------------------------------------------------------------------
def clean_polymer_smiles(smi):
    smi = str(smi).strip()
    for tag in ["[]", "[e]", "[d]", "[t]", "[g]"]:
        smi = smi.replace(tag, "*")
    if "*" not in smi:
        smi = f"*{smi}*"
    return smi

def star_substitution_3d(smi):
    cleaned = clean_polymer_smiles(smi)
    return cleaned.replace("*", "[Fr]")

def mol_from_smiles(smi, mode="2d"):
    if mode == "3d":
        smi_parsed = star_substitution_3d(smi)
    else:
        smi_parsed = clean_polymer_smiles(smi)
        
    for s in (smi_parsed, smi_parsed.replace("[Fr]", "C"), smi_parsed.replace("*", "C"), smi_parsed.replace("*", "[H]")):
        m = Chem.MolFromSmiles(s)
        if m is not None:
            return m
    return None

def canonicalize_smiles(smi):
    mol = mol_from_smiles(smi, mode="2d")
    if mol is not None:
        return Chem.MolToSmiles(mol, canonical=True)
    return str(smi)

def prep_wide_format(df):
    df_clean = df.copy()
    df_clean['smiles_canon'] = df_clean['smiles'].apply(canonicalize_smiles)
    df_clean['target_type'] = df_clean['target_type'].str.lower()
    
    wide = []
    for smi, group in tqdm(df_clean.groupby('smiles_canon'), desc="Pivoting Dataset"):
        row = {'smiles_canon': smi}
        for t in TARGETS:
            if 'target' in group.columns:
                val = group[group['target_type'] == t]['target']
                if len(val) > 0:
                    row[t] = val.mean()
                else:
                    row[t] = np.nan
            else:
                row[t] = np.nan
        wide.append(row)
    return pd.DataFrame(wide)

def mol_dimer(smi):
    mol = mol_from_smiles(smi)
    if mol is None: return None
    dummy_idx = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
    if len(dummy_idx) != 2: return None
    try:
        head, tail = dummy_idx[0], dummy_idx[1]
        head_nbrs = list(mol.GetAtomWithIdx(head).GetNeighbors())
        tail_nbrs = list(mol.GetAtomWithIdx(tail).GetNeighbors())
        if len(head_nbrs) != 1 or len(tail_nbrs) != 1: return None
        combo = Chem.CombineMols(mol, mol)
        rw = Chem.RWMol(combo)
        n = mol.GetNumAtoms()
        rw.AddBond(tail_nbrs[0].GetIdx(), head_nbrs[0].GetIdx() + n, Chem.BondType.SINGLE)
        for idx in sorted([tail, head + n], reverse=True): rw.RemoveAtom(idx)
        out = rw.GetMol()
        Chem.SanitizeMol(out)
        return out
    except Exception: return None

# -----------------------------------------------------------------------------
# 3. POLYMER GENOME & 3D DESCRIPTORS
# -----------------------------------------------------------------------------
def descriptor_vector(mol):
    if mol is None: return np.zeros(56, dtype=np.float32)
    vals = []
    fns = [
        Descriptors.MolWt, Descriptors.HeavyAtomMolWt, Descriptors.ExactMolWt, Descriptors.MolLogP, 
        Descriptors.MolMR, Descriptors.TPSA, Descriptors.NumValenceElectrons, Descriptors.NumRadicalElectrons, 
        Descriptors.FractionCSP3, Descriptors.HeavyAtomCount, Descriptors.NHOHCount, Descriptors.NOCount, 
        Descriptors.NumHAcceptors, Descriptors.NumHDonors, Descriptors.NumHeteroatoms, Descriptors.NumRotatableBonds, 
        Descriptors.RingCount, Descriptors.BalabanJ, Descriptors.BertzCT, Descriptors.Chi0, Descriptors.Chi0n, 
        Descriptors.Chi0v, Descriptors.Chi1, Descriptors.Chi1n, Descriptors.Chi1v, Descriptors.Chi2n, 
        Descriptors.Chi2v, Descriptors.Chi3n, Descriptors.Chi3v, Descriptors.Chi4n, Descriptors.Chi4v, 
        Descriptors.Kappa1, Descriptors.Kappa2, Descriptors.Kappa3, Descriptors.LabuteASA, 
        Descriptors.PEOE_VSA1, Descriptors.PEOE_VSA2, Descriptors.PEOE_VSA6, Descriptors.PEOE_VSA7, 
        Descriptors.PEOE_VSA8, Descriptors.SMR_VSA1, Descriptors.SMR_VSA3, Descriptors.SMR_VSA5, 
        Descriptors.SlogP_VSA1, Descriptors.SlogP_VSA2, Descriptors.SlogP_VSA3, Descriptors.SlogP_VSA5, 
        Descriptors.SlogP_VSA6, EState_VSA.EState_VSA1, EState_VSA.EState_VSA2, EState_VSA.EState_VSA3, 
        EState_VSA.EState_VSA4, EState_VSA.EState_VSA5, EState_VSA.EState_VSA6, EState_VSA.EState_VSA7, 
        EState_VSA.EState_VSA8
    ]
    for fn in fns:
        try: 
            val = fn(mol)
            vals.append(float(val) if np.isfinite(val) else 0.0)
        except: vals.append(0.0)
    return np.asarray(vals, dtype=np.float32)

def gasteiger_features(mol):
    if mol is None: return np.zeros(10, dtype=np.float32)
    try:
        mc = Chem.RWMol(mol)
        AllChem.ComputeGasteigerCharges(mc)
        c = [float(a.GetDoubleProp("_GasteigerCharge")) for a in mc.GetAtoms() if a.HasProp("_GasteigerCharge")]
        c = np.array(c); c = c[np.isfinite(c)]
        if len(c) == 0: return np.zeros(10, dtype=np.float32)
        pos = c[c > 0]; neg = c[c < 0]
        return np.array([
            c.mean(), c.std(), c.min(), c.max(), 
            pos.sum() if len(pos) > 0 else 0.0, neg.sum() if len(neg) > 0 else 0.0, 
            len(pos)/len(c), len(neg)/len(c), np.abs(c).mean(), c.max() - c.min()
        ], dtype=np.float32)
    except: return np.zeros(10, dtype=np.float32)

def get_topology_features(mol):
    if mol is None: return np.zeros(242, dtype=np.float32)
    try: ac = np.array(rdMolDescriptors.CalcAUTOCORR2D(mol), dtype=np.float32)
    except: ac = np.zeros(192, dtype=np.float32)
    try: mqn = np.array(rdMolDescriptors.CalcMQNs(mol), dtype=np.float32)
    except: mqn = np.zeros(42, dtype=np.float32)
    try:
        idx = np.array(ES.EStateIndices(mol), dtype=np.float32); idx = idx[np.isfinite(idx)]
        est = np.array([idx.mean(), idx.std(), idx.min(), idx.max(), (idx>0).mean(), (idx<0).mean(), np.abs(idx).mean(), idx.max()-idx.min()]) if len(idx)>0 else np.zeros(8)
    except: est = np.zeros(8, dtype=np.float32)
    return np.concatenate([ac, mqn, est])

def get_dimer_deltas(mol_m, mol_d):
    if mol_m is None or mol_d is None: return np.zeros(30, dtype=np.float32)
    def _counts(m): return np.array([Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m), Descriptors.NumRotatableBonds(m), Descriptors.RingCount(m), rdMolDescriptors.CalcNumAromaticRings(m), rdMolDescriptors.CalcNumAliphaticRings(m), rdMolDescriptors.CalcNumAmideBonds(m), Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m), Descriptors.NumHeteroatoms(m), Crippen.MolMR(m), Descriptors.BertzCT(m), Descriptors.HeavyAtomCount(m), Descriptors.FractionCSP3(m)], dtype=np.float32)
    try: 
        cm = _counts(mol_m); cd = _counts(mol_d)
        return np.concatenate([cd - (2.0 * cm), cd / np.clip(cm * 2.0, 1e-6, None)])
    except: return np.zeros(30, dtype=np.float32)

def backbone_sidechain_features(mol):
    if mol is None: return np.zeros(10, dtype=np.float32)
    try:
        dummy = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() == 0]
        heavy = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() > 1]
        if len(dummy) < 2: return np.array([0]*9 + [len(dummy)], dtype=np.float32)
        path = list(Chem.GetShortestPath(mol, dummy[0], dummy[1]))
        bb = set(path) - set(dummy); nb = max(1, len(bb)); ns = max(0, max(1, len(heavy)) - nb)
        be = sum(1 for i in bb for n in mol.GetAtomWithIdx(i).GetNeighbors() if n.GetIdx() not in bb and n.GetIdx() not in dummy)
        ar = sum(1 for i in bb if mol.GetAtomWithIdx(i).GetIsAromatic()); ht = sum(1 for i in bb if mol.GetAtomWithIdx(i).GetAtomicNum() not in (1,6))
        return np.asarray([nb, ns, ns/max(1, len(heavy)), ar/nb, ht/nb, be, be/nb, nb/max(1, len(heavy)), len(path), len(dummy)], dtype=np.float32)
    except: return np.zeros(10, dtype=np.float32)

class _Embed3DTimeout(Exception): pass
def _embed3d_alarm_handler(s, f): raise _Embed3DTimeout()

def compute_3d_features_safe(mol):
    if not COMPUTE_3D or mol is None or mol.GetNumHeavyAtoms() > MAX_HEAVY_FOR_3D: return np.zeros(15, dtype=np.float32)
    has_signal = hasattr(signal, 'SIGALRM')
    if has_signal:
        old_h = signal.signal(signal.SIGALRM, _embed3d_alarm_handler)
        signal.alarm(5)
    try:
        m = Chem.AddHs(mol); p = AllChem.ETKDGv3(); p.randomSeed = SEED; p.useRandomCoords = True
        cid = AllChem.EmbedMolecule(m, p)
        if cid < 0: cid = AllChem.EmbedMolecule(m, useRandomCoords=True, randomSeed=SEED, maxAttempts=10)
        if cid < 0: return np.zeros(15, dtype=np.float32)
        try:
            if AllChem.MMFFOptimizeMolecule(m, maxIters=50) != 0: AllChem.UFFOptimizeMolecule(m, maxIters=50)
        except: pass
        sv = []
        for fn in [Descriptors3D.PMI1, Descriptors3D.RadiusOfGyration, Descriptors3D.Asphericity, Descriptors3D.Eccentricity]:
            try: sv.append(float(fn(m)) if np.isfinite(fn(m)) else 0.0)
            except: sv.append(0.0)
        try: vol = float(AllChem.ComputeMolVolume(m))
        except: vol = 0.0
        return np.array(sv + [vol] + [0] * 10, dtype=np.float32)[:15]
    except: return np.zeros(15, dtype=np.float32)
    finally:
        if has_signal:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old_h)

# -----------------------------------------------------------------------------
# 4. DEEP TRI-MODAL ARCHITECTURE (CHARACTER SEQUENCING & GRAPH)
# -----------------------------------------------------------------------------
SMILES_CHARS = list(" #%()+-./0123456789=@ABCDEFGHIKLMNOPRSTVXZ[\\]abcdefgilmnoprstuy*")
CHAR2ID = {c: i + 1 for i, c in enumerate(SMILES_CHARS)}

def encode_smiles(smi_list, max_len=256):
    out = np.zeros((len(smi_list), max_len), dtype=np.int64)
    for i, s in enumerate(smi_list):
        ids = [CHAR2ID.get(c, 0) for c in str(s)[:max_len]]
        out[i, :len(ids)] = ids
    return out

def mol_to_graph(mol, max_atomic_num=100):
    if mol is None or mol.GetNumAtoms() == 0: return (np.zeros((1, 7), dtype=np.float32), np.zeros((2, 1), dtype=np.int64), np.zeros((1, 6), dtype=np.float32))
    af = []
    for a in mol.GetAtoms():
        hyb = a.GetHybridization()
        af.append([min(a.GetAtomicNum(), max_atomic_num), a.GetDegree(), a.GetFormalCharge(), a.GetTotalNumHs(), float(a.GetIsAromatic()), float(a.IsInRing()), float(int(hyb)) if hyb is not None else 0.0])
    src, dst, ea = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bt = b.GetBondTypeAsDouble()
        feat = [float(bt == 1.0), float(bt == 2.0), float(bt == 3.0), float(bt == 1.5), float(b.GetIsConjugated()), float(b.IsInRing())]
        src.append(i); dst.append(j); ea.append(feat)
        src.append(j); dst.append(i); ea.append(feat)
    if not src: src, dst, ea = [0], [0], [[0.0] * 6]
    return np.array(af, dtype=np.float32), np.array([src, dst], dtype=np.int64), np.array(ea, dtype=np.float32)

def collate_graphs(graph_list, device):
    af_l, ei_l, ea_l, b_l = [], [], [], []
    off = 0
    for gi, (af, ei, ea) in enumerate(graph_list):
        n = af.shape[0]; af_l.append(af); ei_l.append(ei + off); ea_l.append(ea); b_l.append(np.full(n, gi, dtype=np.int64)); off += n
    return torch.tensor(np.vstack(af_l), dtype=torch.float32, device=device), torch.tensor(np.hstack(ei_l), dtype=torch.long, device=device), torch.tensor(np.vstack(ea_l), dtype=torch.float32, device=device), torch.tensor(np.concatenate(b_l), dtype=torch.long, device=device)

class MultiTaskUncertaintyLoss(nn.Module):
    def __init__(self, num_tasks):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))
    def forward(self, preds, targets, masks):
        loss = 0
        for i in range(preds.shape[1]):
            valid_mask = masks[:, i].bool()
            if valid_mask.sum() > 0:
                mse = F.mse_loss(preds[valid_mask, i], targets[valid_mask, i])
                loss += 0.5 * torch.exp(-self.log_vars[i]) * mse + 0.5 * self.log_vars[i]
        return loss

class MultiTaskCharCNN(nn.Module):
    def __init__(self, vocab_size, emb_dim=48, ch=96, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(emb_dim, ch, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(ch, ch, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(ch, hidden, batch_first=True, bidirectional=True)
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1)) for _ in range(NUM_TARGETS)])
    def forward(self, x):
        mask = (x != 0).float().unsqueeze(1)
        e = self.emb(x).transpose(1, 2)
        c = F.relu(self.conv2(F.relu(self.conv1(e)) * mask)) * mask
        out, _ = self.lstm(c.transpose(1, 2))
        pooled = (out * mask.transpose(1, 2)).sum(dim=1) / mask.sum(dim=2).clamp(min=1e-9)
        return torch.cat([head(pooled) for head in self.heads], dim=1), pooled

class SMILESPPDCPOA_1DCNN_GRU(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, ch=128, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(emb_dim, ch, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(ch, ch, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        self.gru = nn.GRU(ch, hidden, batch_first=True, dropout=0.2, bidirectional=True)
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1)) for _ in range(NUM_TARGETS)])
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        c1 = self.pool1(F.relu(self.conv1(e)))
        c2 = self.pool2(F.relu(self.conv2(c1)))
        out, _ = self.gru(c2.transpose(1, 2))
        pooled = out.mean(dim=1)
        return torch.cat([head(pooled) for head in self.heads], dim=1), pooled

class GINLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(nn.Linear(dim + 6, dim), nn.ReLU())
        self.update_mlp = nn.Sequential(nn.Linear(dim * 2, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.norm = nn.LayerNorm(dim)
    def forward(self, x, ei, ea):
        msg = self.edge_mlp(torch.cat([x[ei[0]], ea], dim=-1))
        agg = torch.zeros(x.size(0), msg.size(-1), device=x.device).index_add_(0, ei[1], msg)
        return self.norm(x + self.update_mlp(torch.cat([x, agg], dim=-1)))

class MultiTaskMolGNN(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(101, hidden)
        self.proj = nn.Sequential(nn.Linear(hidden + 6, hidden), nn.ReLU())
        self.layers = nn.ModuleList([GINLayer(hidden) for _ in range(3)])
        self.heads = nn.ModuleList([nn.Sequential(nn.Linear(hidden * 2, 64), nn.ReLU(), nn.Linear(64, 1)) for _ in range(NUM_TARGETS)])
    def forward(self, af, ei, ea, batch, n_graphs):
        x = self.proj(torch.cat([self.emb(af[:, 0].long()), af[:, 1:]], dim=-1))
        for layer in self.layers: x = layer(x, ei, ea)
        sum_ = torch.zeros(n_graphs, x.size(-1), device=x.device).index_add_(0, batch, x)
        cnt = torch.zeros(n_graphs, 1, device=x.device).index_add_(0, batch, torch.ones(x.size(0), 1, device=x.device)).clamp(min=1)
        mean_p = sum_ / cnt
        max_p = torch.full((n_graphs, x.size(-1)), -1e9, device=x.device).scatter_reduce_(0, batch.unsqueeze(1).expand(-1, x.size(-1)), x, reduce='amax', include_self=False)
        pooled = torch.cat([mean_p, max_p], dim=1)
        return torch.cat([head(pooled) for head in self.heads], dim=1), pooled

def pareto_blend_weights(oof_preds, y_true):
    best_weights = np.zeros(oof_preds.shape[1]); best_weights[0] = 1.0
    for _ in range(250):
        best_score = -np.inf; best_idx = -1
        for j in range(oof_preds.shape[1]):
            tw = best_weights.copy(); tw[j] += 0.05; tw /= tw.sum()
            score = r2_score(y_true, np.average(oof_preds, axis=1, weights=tw))
            if score > best_score: best_score = score; best_idx = j
        if best_idx != -1:
            best_weights[best_idx] += 0.05; best_weights /= best_weights.sum()
    return best_weights

# -----------------------------------------------------------------------------
# 5. TITAN OMNI EXECUTION
# -----------------------------------------------------------------------------
def main():
    print(f"[1/7] System Initialized (High Purity Mode). Target Device: {TORCH_DEVICE}")
    data_dir = Path("/kaggle/input/competitions/ppp-round-2")
    if not data_dir.exists(): data_dir = Path(".")
    
    train_long = pd.read_csv(data_dir / "train.csv")
    test_long  = pd.read_csv(data_dir / "test.csv")
    
    print("\n--- Pivoting to Multi-Task Wide Format ---")
    train_wide = prep_wide_format(train_long)
    test_wide = prep_wide_format(test_long)
    
    # WE REMOVED PI1M INJECTION TO PREVENT LABEL NOISE AND PROTECT THE 0.881 BASELINE
    all_smiles = np.concatenate([train_wide['smiles_canon'].values, test_wide['smiles_canon'].values])
    n_train = len(train_wide)
    
    print("\n[2/7] Extracting Polymer Genome & 3D Star Substitution Geometries...")
    mfp_r3 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=N_BITS)
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=N_BITS)
    
    fp_l, ap_l, mc_l, desc_l, topo_l, delta_l, bbsc_l, shp_l, gast_l, graphs = [], [], [], [], [], [], [], [], [], []
    
    for smi in tqdm(all_smiles, desc="Physics & Graphs"):
        m = mol_from_smiles(smi, mode="2d"); d = mol_dimer(smi)
        if m is None:
            fp_l.append(np.zeros(N_BITS, dtype=np.float32)); ap_l.append(np.zeros(N_BITS, dtype=np.float32))
            mc_l.append(np.zeros(167, dtype=np.float32)); desc_l.append(np.zeros(56, dtype=np.float32))
            topo_l.append(np.zeros(242, dtype=np.float32)); delta_l.append(np.zeros(30, dtype=np.float32))
            bbsc_l.append(np.zeros(10, dtype=np.float32)); shp_l.append(np.zeros(15, dtype=np.float32))
            gast_l.append(np.zeros(10, dtype=np.float32)); graphs.append(mol_to_graph(m))
            continue
            
        r3 = np.zeros(N_BITS, dtype=np.float32); DataStructs.ConvertToNumpyArray(mfp_r3.GetFingerprint(m), r3); fp_l.append(r3)
        ap = np.zeros(N_BITS, dtype=np.float32); DataStructs.ConvertToNumpyArray(ap_gen.GetFingerprint(m), ap); ap_l.append(ap)
        mc = np.zeros(167, dtype=np.float32); DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(m), mc); mc_l.append(mc)
        desc_l.append(descriptor_vector(m)); topo_l.append(get_topology_features(m))
        delta_l.append(get_dimer_deltas(m, d)); bbsc_l.append(backbone_sidechain_features(m))
        shp_l.append(compute_3d_features_safe(mol_from_smiles(smi, mode="3d"))); gast_l.append(gasteiger_features(m))
        graphs.append(mol_to_graph(m))

    FP, AP, MC, DESC, TOPO = np.vstack(fp_l).astype(np.float32), np.vstack(ap_l).astype(np.float32), np.vstack(mc_l).astype(np.float32), np.vstack(desc_l).astype(np.float32), np.vstack(topo_l).astype(np.float32)
    DELTA, BBSC, SHP, GAST = np.vstack(delta_l).astype(np.float32), np.vstack(bbsc_l).astype(np.float32), np.vstack(shp_l).astype(np.float32), np.vstack(gast_l).astype(np.float32)
    
    tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2,6), min_df=2, max_features=15000, dtype=np.float32)
    TXT = TruncatedSVD(n_components=200, random_state=SEED).fit_transform(tfidf.fit_transform(all_smiles)).astype(np.float32)

    print("\n[3/7] Tri-Modal Deep Learning Embeddings (CharCNN, MolGNN, 1DCNN-GRU)...")
    y_raw_train = train_wide[TARGETS].values.astype(np.float32)
    y_scaled = np.full_like(y_raw_train, np.nan)
    y_lsn = np.full_like(y_raw_train, np.nan)
    lsn_scaler = MinMaxScaler(feature_range=(0, 1))
    
    for i in range(NUM_TARGETS):
        mask = ~np.isnan(y_raw_train[:, i]); sc = RobustScaler()
        if mask.sum() > 0:
            y_scaled[mask, i] = sc.fit_transform(y_raw_train[mask, i].reshape(-1, 1)).flatten()
            y_lsn[mask, i] = lsn_scaler.fit_transform(y_raw_train[mask, i].reshape(-1, 1)).flatten()
        
    X_char = encode_smiles(all_smiles)
    
    # EPOCH UPGRADE: Increased to 75 epochs for deeper convergence
    def train_dl(model, y_target, lr=1e-3, epochs=75):
        crit = MultiTaskUncertaintyLoss(NUM_TARGETS).to(TORCH_DEVICE)
        opt = torch.optim.AdamW(list(model.parameters()) + list(crit.parameters()), lr=lr)
        model.train()
        for ep in range(epochs):
            perm = torch.randperm(n_train)
            for i in range(0, n_train, 128):
                idx = perm[i:i+128].numpy()
                bx = torch.tensor(X_char[idx], dtype=torch.long, device=TORCH_DEVICE)
                by = torch.tensor(np.nan_to_num(y_target[idx], nan=0.0), dtype=torch.float32, device=TORCH_DEVICE)
                bm = torch.tensor(~np.isnan(y_target[idx]), dtype=torch.float32, device=TORCH_DEVICE)
                opt.zero_grad(); preds, _ = model(bx); loss = crit(preds, by, bm); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        
        model.eval(); embs = []
        with torch.no_grad():
            for i in range(0, len(X_char), 128):
                _, emb = model(torch.tensor(X_char[i:i+128], dtype=torch.long, device=TORCH_DEVICE))
                embs.append(emb.cpu().numpy())
        return np.vstack(embs)
    
    print(" -> Training CharCNN (75 Epochs)...")
    char_emb = train_dl(MultiTaskCharCNN(vocab_size=len(SMILES_CHARS)).to(TORCH_DEVICE), y_scaled)
    print(" -> Training SMILES-PPDCPOA (75 Epochs)...")
    ppdcpoa_emb = train_dl(SMILESPPDCPOA_1DCNN_GRU(vocab_size=len(SMILES_CHARS)).to(TORCH_DEVICE), y_lsn)
    
    print(" -> Training MolGNN (75 Epochs)...")
    gnn_model = MultiTaskMolGNN().to(TORCH_DEVICE)
    crit_gnn = MultiTaskUncertaintyLoss(NUM_TARGETS).to(TORCH_DEVICE)
    opt_gnn = torch.optim.AdamW(list(gnn_model.parameters()) + list(crit_gnn.parameters()), lr=2e-3)
    
    gnn_model.train()
    for ep in range(75):
        perm = torch.randperm(n_train)
        for i in range(0, n_train, 128):
            idx = perm[i:i+128].numpy()
            sub = [graphs[j] for j in idx]; af, ei, ea, batch = collate_graphs(sub, TORCH_DEVICE)
            yb = torch.tensor(np.nan_to_num(y_scaled[idx], nan=0.0), dtype=torch.float32, device=TORCH_DEVICE)
            mb = torch.tensor(~np.isnan(y_scaled[idx]), dtype=torch.float32, device=TORCH_DEVICE)
            opt_gnn.zero_grad(); preds, _ = gnn_model(af, ei, ea, batch, len(idx))
            loss = crit_gnn(preds, yb, mb); loss.backward(); torch.nn.utils.clip_grad_norm_(gnn_model.parameters(), 1.0); opt_gnn.step()
            
    gnn_model.eval(); gnn_embs = []
    with torch.no_grad():
        for i in range(0, len(graphs), 128):
            sub = [graphs[j] for j in range(i, min(i+128, len(graphs)))]
            af, ei, ea, batch = collate_graphs(sub, TORCH_DEVICE)
            _, emb = gnn_model(af, ei, ea, batch, len(sub)); gnn_embs.append(emb.cpu().numpy())
    gnn_emb = np.vstack(gnn_embs)
    
    print("\n[4/7] Partitioning Hybrid Representation Matrices...")
    feature_sets = {
        "count_text": np.hstack([MC, DESC, TXT]),
        "fp_all": np.hstack([FP, AP, MC, DESC]),
        "flagship": np.hstack([FP, MC, DESC, TOPO, DELTA, BBSC, GAST, char_emb, ppdcpoa_emb, gnn_emb]),
        "research3d": np.hstack([FP, MC, DESC, TOPO, DELTA, BBSC, SHP]),
    }
    for k in feature_sets.keys():
        cl = VarianceThreshold(threshold=1e-5).fit_transform(SimpleImputer(strategy='median').fit_transform(feature_sets[k]))
        feature_sets[k] = {'train': cl[:n_train], 'test': cl[n_train:]}

    final_wide_preds = test_wide[['smiles_canon']].copy()
    
    for i, t in enumerate(TARGETS):
        print(f"\n{'='*60}\nEnsemble Training for: {t.upper()}\n{'='*60}")
        valid_mask = ~np.isnan(y_raw_train[:, i])
        y_t = y_raw_train[valid_mask, i]
        
        use_qt = t in ['tg', 'eps', 'eea', 'nc']
        qt = QuantileTransformer(output_distribution='normal', random_state=SEED) if use_qt else None
        y_fit = qt.fit_transform(y_t.reshape(-1, 1)).flatten() if use_qt else y_t
        
        models = [
            ("ridge_count", "count_text", make_pipeline(RobustScaler(), Ridge(alpha=10.0))),
            ("elastic_count", "count_text", make_pipeline(RobustScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5))),
            ("lgb_fpall", "fp_all", lgb.LGBMRegressor(n_estimators=1800, learning_rate=0.025, random_state=SEED, verbose=-1)),
            ("lgb_flagship", "flagship", lgb.LGBMRegressor(n_estimators=1800, learning_rate=0.025, random_state=SEED, verbose=-1)),
            ("lgb_flagship_hbr", "flagship", lgb.LGBMRegressor(objective="huber", alpha=0.9, n_estimators=1800, learning_rate=0.025, random_state=SEED, verbose=-1)),
            ("lgb_flagship_qtl", "flagship", lgb.LGBMRegressor(objective="quantile", alpha=0.5, n_estimators=1800, learning_rate=0.025, random_state=SEED, verbose=-1)),
            ("lgb_research3d", "research3d", lgb.LGBMRegressor(n_estimators=1800, learning_rate=0.025, random_state=SEED, verbose=-1)),
            ("xgb_master", "flagship", xgb.XGBRegressor(n_estimators=1800, learning_rate=0.025, max_depth=6, random_state=SEED, tree_method="hist")),
            ("cat_master", "flagship", CatBoostRegressor(iterations=1800, learning_rate=0.025, depth=6, verbose=False, random_seed=SEED)),
        ]
        
        oof_matrix = np.zeros((len(y_fit), len(models)))
        test_matrix = np.zeros((len(test_wide), len(models)))
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        
        for m_idx, (m_name, f_set, model) in enumerate(models):
            X_tr = feature_sets[f_set]['train'][valid_mask]
            X_te = feature_sets[f_set]['test']
            test_accum = np.zeros(len(X_te))
            for tr_idx, va_idx in kf.split(X_tr):
                xtr, xva = X_tr[tr_idx], X_tr[va_idx]
                ytr, yva = y_fit[tr_idx], y_fit[va_idx]
                if "lgb" in m_name: model.fit(xtr, ytr, eval_set=[(xva, yva)], callbacks=[lgb.early_stopping(30, verbose=False)])
                elif "cat" in m_name: model.fit(xtr, ytr, eval_set=(xva, yva), early_stopping_rounds=30)
                elif "xgb" in m_name: model.fit(xtr, ytr, eval_set=[(xva, yva)], verbose=False)
                else: model.fit(xtr, ytr)
                oof_matrix[va_idx, m_idx] = model.predict(xva)
                test_accum += model.predict(X_te) / float(N_SPLITS)
            test_matrix[:, m_idx] = test_accum
            print(f"  {m_name:22s} OOF R2={r2_score(y_fit, oof_matrix[:, m_idx]):.5f}")
            
        best_w = pareto_blend_weights(oof_matrix, y_fit)
        blend_test = np.average(test_matrix, axis=1, weights=best_w)
        
        print("  100% Full-Data Refit...")
        refit_test = np.zeros_like(blend_test)
        for m_idx, (m_name, f_set, model) in enumerate(models):
            if best_w[m_idx] > 0.01:
                model.fit(feature_sets[f_set]['train'][valid_mask], y_fit)
                refit_test += model.predict(feature_sets[f_set]['test']) * best_w[m_idx]
                
        # --- NEW VARIANCE-BASED CONFIDENCE PSEUDO-LABELING ---
        print("  Variance-Based Test-Time Augmentation (Top 10% Agreement)...")
        active_models = test_matrix[:, best_w > 0.01]
        test_variance = np.std(active_models, axis=1)
        conf_mask = test_variance <= np.percentile(test_variance, 10)  # GUARANTEED to capture test data where models agree perfectly
        
        X_tr_base = feature_sets["flagship"]['train'][valid_mask]
        X_test_conf = feature_sets["flagship"]['test'][conf_mask]
        X_aug = np.vstack([X_tr_base, X_test_conf])
        y_aug = np.concatenate([y_fit, refit_test[conf_mask]])
        
        ps_model = lgb.LGBMRegressor(n_estimators=2500, learning_rate=0.015, num_leaves=31, random_state=SEED, verbose=-1)
        ps_model.fit(X_aug, y_aug)
        final_preds = (0.7 * refit_test) + (0.3 * ps_model.predict(feature_sets["flagship"]['test']))
        
        final_wide_preds[t] = qt.inverse_transform(final_preds.reshape(-1, 1)).flatten() if use_qt else final_preds
    
    print("\n[5/7] Applying Immutable Thermodynamic Constraints...")
    final_wide_preds['nc'] = np.clip(final_wide_preds['nc'], 1.0, None)
    final_wide_preds['egc'] = np.clip(final_wide_preds['egc'], 0.0, None)
    final_wide_preds['egb'] = np.clip(final_wide_preds['egb'], 0.0, None)
    final_wide_preds['eps'] = np.maximum(final_wide_preds['eps'], final_wide_preds['nc']**2)
    final_wide_preds['eea'] = np.minimum(final_wide_preds['eea'], final_wide_preds['ei'])
    
    print("\n[6/7] Exporting High-Purity Submission...")
    pred_dict = final_wide_preds.set_index('smiles_canon').to_dict('index')
    raw_to_canon = {smi: canonicalize_smiles(smi) for smi in test_long['smiles'].unique()}
    
    pd.DataFrame({
        'id': test_long['id'], 
        'target': [pred_dict[raw_to_canon[row['smiles']]][row['target_type'].lower()] for _, row in test_long.iterrows()]
    }).to_csv("submission.csv", index=False)
    print("\n✅ Pure Titan Omni Pipeline Completed. Ready for Leaderboard!")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 50.8 MB/s eta 0:00:00
[1/7] System Initialized (High Purity Mode). Target Device: cuda

--- Pivoting to Multi-Task Wide Format ---


Pivoting Dataset:   0%|          | 0/5920 [00:00<?, ?it/s]

Pivoting Dataset:   0%|          | 0/4133 [00:00<?, ?it/s]


[2/7] Extracting Polymer Genome & 3D Star Substitution Geometries...


Physics & Graphs:   0%|          | 0/10053 [00:00<?, ?it/s]


[3/7] Tri-Modal Deep Learning Embeddings (CharCNN, MolGNN, 1DCNN-GRU)...
 -> Training CharCNN (75 Epochs)...
 -> Training SMILES-PPDCPOA (75 Epochs)...
 -> Training MolGNN (75 Epochs)...

[4/7] Partitioning Hybrid Representation Matrices...

Ensemble Training for: EGC
  ridge_count            OOF R2=0.84360
  elastic_count          OOF R2=0.76633
  lgb_fpall              OOF R2=0.89414
  lgb_flagship           OOF R2=0.95194
  lgb_flagship_hbr       OOF R2=0.95174
  lgb_flagship_qtl       OOF R2=0.94666
  lgb_research3d         OOF R2=0.88270
  xgb_master             OOF R2=0.94507
  cat_master             OOF R2=0.95940
  100% Full-Data Refit...
  Variance-Based Test-Time Augmentation (Top 10% Agreement)...

Ensemble Training for: EGB
  ridge_count            OOF R2=0.77000
  elastic_count          OOF R2=0.79370
  lgb_fpall              OOF R2=0.87428
  lgb_flagship           OOF R2=0.96565
  lgb_flagship_hbr       OOF R2=0.96527
  lgb_flagship_qtl       OOF R2=0.95140
  lgb_researc